# 2. Calidad de datos

Antes de modelar se revisan los nulos, los duplicados, los valores imposibles o extremos y la consistencia de las categorías.

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from utils import *

set_style()
np.random.seed(SEED)
pd.set_option("display.max_columns", 40)
df = load_data()

## 2.1 Valores nulos

In [ ]:
nulos = df.isna().sum()
nulos = nulos[nulos > 0].to_frame("nulos")
nulos["porcentaje"] = (100 * nulos["nulos"] / len(df)).round(2)
nulos

Solo `area` tiene nulos: **16 329 registros (27.69 %)**. El resto del dataset está completo.

¿Son nulos aleatorios? Se compara el porcentaje de nulos de `area` en cada clase:

In [ ]:
nulos_clase = (df.groupby(TARGET, observed=True)["area"]
               .apply(lambda s: 100 * s.isna().mean())
               .round(1).rename("% nulos en area"))

fig, ax = plt.subplots(figsize=(6, 2.6))
ax.barh(nulos_clase.index.astype(str), nulos_clase.values, height=0.5,
        color=[CLASS_COLORS[c] for c in nulos_clase.index])
for y, v in enumerate(nulos_clase.values):
    ax.text(v + 0.6, y, f"{v:.1f} %", va="center", color="#0b0b0b")
ax.set_xlabel("% de plantas sin área registrada")
ax.set_title("Nulos de area por clase de aptitud")
ax.grid(axis="y", visible=False)
ax.set_xlim(0, 40)
plt.tight_layout(); plt.show()
nulos_clase.to_frame()

```{important}
Los nulos de `area` **no son aleatorios**. Faltan en el 33.1 % de las plantas de aptitud Alta, frente al 12.4 % de Baja y al 11.4 % de Media. Si solo se imputa la mediana, se pierde esa señal. Conviene acompañar la imputación con un **indicador binario `area_missing`**, siempre ajustado dentro del pipeline sobre el conjunto de entrenamiento.
```

## 2.2 Duplicados

In [ ]:
features = NUMERIC + CATEGORICAL
resumen_dup = pd.Series({
    "Filas duplicadas exactas": df.duplicated().sum(),
    "Códigos (code) repetidos": df["code"].duplicated().sum(),
    "Filas con coordenadas repetidas": df.duplicated(["latitude", "longitude"]).sum(),
    "Filas con predictoras + objetivo idénticos": df.duplicated(features + [TARGET]).sum(),
}, name="n")
resumen_dup.to_frame()

In [ ]:
coords = df[df.duplicated(["latitude", "longitude"], keep=False)]
grupos = coords.groupby(["latitude", "longitude"])
print(f"Ubicaciones repetidas: {grupos.ngroups:,} (abarcan {len(coords):,} filas)")
print(f"Ubicaciones repetidas con clases distintas: {(grupos[TARGET].nunique() > 1).sum()}")

No hay filas idénticas ni códigos repetidos: cada registro es una planta distinta. Sin embargo, **6 830 filas repiten las coordenadas de otra planta**: hay 3 064 ubicaciones compartidas que abarcan 9 894 filas, típicamente parques construidos en varias fases, y **3 245 filas son idénticas a otra en las 17 predictoras y la clase**.

Con una partición aleatoria, una copia puede caer en entrenamiento y su gemela en prueba, lo que infla las métricas de prueba. Para la siguiente entrega se recomienda **eliminar esos 3 245 duplicados antes de la partición** o agrupar la validación por coordenada.

## 2.3 Valores extremos e imposibles

In [ ]:
chequeos = pd.Series({
    "ghi = 0 (irradiación nula)": (df["ghi"] == 0).sum(),
    "humidity = 0 %": (df["humidity"] == 0).sum(),
    "wind_speed = 0": (df["wind_speed"] == 0).sum(),
    "elevation < 0 m": (df["elevation"] < 0).sum(),
    "dist_to_road > 100 km": (df["dist_to_road"] > 100_000).sum(),
    "area > 10 km²": (df["area"] > 1e7).sum(),
}, name="registros")
chequeos.to_frame()

In [ ]:
df[["area", "dist_to_road", "elevation"]].describe(percentiles=[.5, .99]).T.round(1)

- **Valores físicamente imposibles (`ghi = 0`, `humidity = 0`):** solo 3 registros cada uno. Probablemente son errores de extracción en celdas sin dato y pueden tratarse como nulos.
- **Elevación negativa (255 plantas):** es plausible en zonas bajo el nivel del mar (Países Bajos, mar Muerto, depresiones costeras), así que se conserva.
- **`area` y `dist_to_road`:** tienen colas extremas. El área llega a 2.3 × 10⁸ m² con mediana de 8 067 m², y la distancia a 2 296 km con mediana de 724 m. El percentil 99 queda muy por debajo del máximo, lo que indica unos pocos errores de geocodificación o polígonos mal asignados. Se recomienda **transformación logarítmica** y, para modelos sensibles a la escala (KNN, SVM), **recorte (winsorización) en el percentil 99**.

## 2.4 Consistencia de categorías

In [ ]:
for c in CATEGORICAL:
    print(f"{c}: {sorted(df[c].unique())}")

Las cuatro categóricas predictoras no tienen errores de escritura ni niveles vacíos. Mezclan español (`slope_type`, `curvature_type`) e inglés (`aspect_type`, `dt_wind`); esto es cosmético y no afecta al one-hot encoding. La columna excluida `operational_status` sí tiene niveles redundantes (`cancelled` y `cancelled - inferred 4 y`), pero no entra al modelo.